In [ ]:
import scanpy as sc
import decoupler as dc
import plotnine as p9
import liana as li
from liana.method.sp import RandomForestModel, LinearModel, RobustLinearModel
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
from liana.method import MistyData, genericMistyData, lrMistyData

In [ ]:
adata_raw = sc.read('DBiT_imputation-CellViT-MALT_spatialDE_all_cells_annotated.h5ad')

In [ ]:
adata_raw

In [ ]:
RNA_names = adata_raw.var_names.to_list()
#RNA_names

In [ ]:
gene_names = [RNA for RNA in RNA_names if RNA[-12:]=='--transcript']
print(len(gene_names))
gene_names_final = [gene[:-12] for gene in gene_names]
#gene_names_final

In [ ]:
exon_names = [RNA for RNA in RNA_names if RNA[-6:]=='--exon']
len(exon_names)

In [ ]:
adata_genes = adata_raw[:, gene_names].copy()
adata_genes.var_names = gene_names_final

In [ ]:
adata_genes.var_names

In [ ]:
adata_genes.X = adata_genes.layers['counts'].copy()
sc.pp.normalize_total(adata_genes, target_sum=1e4)
sc.pp.log1p(adata_genes)

In [ ]:
sc.pl.spatial(adata_genes, color=['celltype'], size=1.3)

In [ ]:
ind_tumor1 = (adata_genes.obs['x'] <= 27500)*(adata_genes.obs['x'] >= 17500)*(adata_genes.obs['y'] <= 10000)
ind_tumor2 = (adata_genes.obs['x'] <= 18000)*(adata_genes.obs['x'] >= 15000)*(adata_genes.obs['y'] <= 14000)*(adata_genes.obs['y'] >= 9000)
ind_tumor3 = (adata_genes.obs['x'] <= 31500)*(adata_genes.obs['x'] >= 27500)*(adata_genes.obs['y'] <= 10000)*(adata_genes.obs['y'] >= 7000)
ind_tumor4 = (adata_genes.obs['x'] <= 32500)*(adata_genes.obs['x'] >= 30000)*(adata_genes.obs['y'] <= 16000)*(adata_genes.obs['y'] >= 10000)

In [ ]:
sc.pl.spatial(adata_genes[ind_tumor1, :], color=['celltype'], size=1.5, legend_loc=None)

In [ ]:
Counter(adata_genes[ind_tumor1, :].obs['celltype'])

In [ ]:
sc.pl.spatial(adata_genes[ind_tumor2, :], color=['celltype'], size=1.5, legend_loc=None)

In [ ]:
sc.pl.spatial(adata_genes[ind_tumor3, :], color=['celltype'], size=1.5, legend_loc=None)

In [ ]:
sc.pl.spatial(adata_genes[ind_tumor4, :], color=['celltype'], size=1.5, legend_loc=None)

In [ ]:
ind_tumor_group = np.array(adata_genes.obs['celltype'])
ind_tumor_group[ind_tumor1] = 'Tumor_1'
ind_tumor_group[ind_tumor2] = 'Tumor_2'
ind_tumor_group[ind_tumor3] = 'Tumor_3'
ind_tumor_group[ind_tumor4] = 'Tumor_4'

adata_genes.obs['roi_regions'] = ind_tumor_group
adata_genes_sub1 = adata_genes[ind_tumor1 + ind_tumor2 + ind_tumor3 + ind_tumor4, :].copy()

# Sample data
data = {
    'Group': np.array(adata_genes_sub1.obs['roi_regions']),
    'Celltype': np.array(adata_genes_sub1.obs['celltype']),
    'Value': [1]*adata_genes_sub1.n_obs
}
df = pd.DataFrame(data)

grouped_sum = df.groupby(['Group', 'Celltype'])['Value'].sum().reset_index()

pivot = grouped_sum.pivot(index='Group', columns='Celltype', values='Value')
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

In [ ]:
pivot_pct.columns

In [ ]:
celltype_palette = dict(zip(pivot_pct.columns, adata_genes.uns['celltype_colors']))

In [ ]:
plt.figure(figsize=(6,20))
pivot_pct.plot(kind='bar', stacked=True, color=adata_genes.uns['celltype_colors'])
plt.ylabel('Percentage (%)')
plt.title('Stacked Percentage Bar Chart')
plt.legend('', frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
ind_tumor1_1 = (adata_genes.obs['x'] <= 22000)*(adata_genes.obs['x'] >= 19000)*(adata_genes.obs['y'] <= 8500)*(adata_genes.obs['y'] >= 6500)
ind_tumor1_3 = (adata_genes.obs['x'] <= 24500)*(adata_genes.obs['x'] >= 23000)*(adata_genes.obs['y'] <= 8500)*(adata_genes.obs['y'] >= 7000)
ind1 = adata_genes.obs['y'] <= ((9500-2900) /(29000 - 19500)*(adata_genes.obs['x'] - 19500.)+2900.)
ind2 = adata_genes.obs['y'] >= ((9500-2900) /(29000 - 19500)*(adata_genes.obs['x'] - 22500.)+2900.)
ind3 = adata_genes.obs['y'] >=((8100-2900)/(30000-34000)*(adata_genes.obs['x'] - 34000.)+2900.)
ind4 = adata_genes.obs['y'] <= ((8100-2900)/(30000-34000)*(adata_genes.obs['x'] - 35500.)+2900.)

ind_tumor3_1 = ind1*ind2*ind3*ind4

In [ ]:
ind_tumor_rois = np.array(adata_genes.obs['celltype'])
ind_tumor_rois[ind_tumor1_1] = 'T3'
ind_tumor_rois[ind_tumor1_3] = 'T1'
ind_tumor_rois[ind_tumor3_1] = 'T2'
adata_genes.obs['sub_roi_regions'] = ind_tumor_rois
adata_genes_sub2 = adata_genes[ind_tumor1_1 + ind_tumor1_3 + ind_tumor3_1, :].copy() # + ind_border

# Sample data
data = {
    'Group': np.array(adata_genes_sub2.obs['sub_roi_regions']),
    'Celltype': np.array(adata_genes_sub2.obs['celltype']),
    'Value': [1]*adata_genes_sub2.n_obs
}
df = pd.DataFrame(data)

grouped_sum = df.groupby(['Group', 'Celltype'])['Value'].sum().reset_index()

pivot = grouped_sum.pivot(index='Group', columns='Celltype', values='Value')
pivot = pivot.fillna(0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

In [ ]:
pivot.fillna(0)

In [ ]:
## figure 4D
pivot_pct.plot(kind='bar', stacked=True, color=adata_genes.uns['celltype_colors'])
plt.ylabel('Percentage (%)')
plt.title('Stacked Percentage Bar Chart')
plt.legend('', frameon=False)
#plt.savefig('figure_c2_5.pdf')
plt.show()

In [ ]:
## figure 4D modified
# Extract colors from the dictionary in the order of the columns in the dataframe
plot_colors = [celltype_colors_dict[col] for col in pivot_pct.columns]

ax = pivot_pct.plot(kind='bar', stacked=True, color=plot_colors, figsize=(10, 7))
# Add percentage labels to each bar segment
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 1: 
        x, y = p.get_xy() 
        ax.text(x + width/2, 
                y + height/2, 
                f'{height:.1f}%', 
                horizontalalignment='center', 
                verticalalignment='center',
                fontsize=15,
                color='white')

# Remove frame and grid lines
# Remove only the top and right frame lines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.grid(False)
ax.tick_params(axis='both', which='both', length=0) # Removes the small tick marks

plt.ylabel('Percentage (%)', fontsize=20)
#plt.title('Stacked Percentage Bar Chart')

# Move legend to the right
plt.legend(title='Celltype', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, fontsize=15) # frameon=False removes legend border

plt.tight_layout()
plt.show()

In [ ]:
## figure 4D modified - Horizontal
# Extract colors from the dictionary in the order of the columns in the dataframe
plot_colors = [celltype_colors_dict[col] for col in pivot_pct.columns]

# Change kind to 'barh' for horizontal
ax = pivot_pct.plot(kind='barh', stacked=True, color=plot_colors, figsize=(25, 7))

# Add percentage labels to each bar segment
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if width > 4: # For barh, width represents the percentage value
        x, y = p.get_xy() 
        ax.text(x + width/2, 
                y + height/2, 
                f'{width:.1f}%', # Use width for the label
                horizontalalignment='center', 
                verticalalignment='center',
                fontsize=20,
                color='white')

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Remove grid lines
ax.grid(False)
ax.tick_params(axis='both', which='both', length=0, labelsize=20) 

# Swap labels for horizontal orientation
plt.xlabel('Percentage (%)', fontsize=25)
plt.ylabel('Tumor', fontsize=25) # Update this based on your index name

# Move legend to the right
plt.legend(title='', bbox_to_anchor=(1.0, 1), loc='upper left', frameon=False, fontsize=25, ncol=2)

plt.tight_layout()
plt.show()

### TF scoring

In [ ]:
collectri = dc.op.collectri(organism="human")
collectri

In [ ]:
dc.mt.ulm(data=adata_genes, net=collectri, layer='counts')

In [ ]:
score = dc.pp.get_obsm(adata=adata_genes, key="score_ulm")
score

In [ ]:
score.var_names

In [ ]:
df = dc.tl.rankby_group(adata=score, groupby="celltype", reference="rest", method="t-test_overestim_var")
df = df[df["stat"] > 0.0]
df

In [ ]:
n_markers = 5
source_markers = (
    df.groupby("group")
    .head(n_markers)
    .drop_duplicates("name")
    .groupby("group")["name"]
    .apply(lambda x: list(x))
    .to_dict()
)
source_markers

In [ ]:
sc.set_figure_params(fontsize=25)
sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby="celltype",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
)

In [ ]:
adata_genes

In [ ]:
sc.set_figure_params(
    #figsize=(6, 8), 
    dpi=300, 
    dpi_save=300, 
    frameon=False
)
plt.rcParams['axes.facecolor'] = 'black'
sc.pl.spatial(adata_raw, color='EP300--transcript', layer='counts', vmax=1.2, img_key='hires', size=1.2, alpha_img=1.0)

In [ ]:
ind_tumor2 = (adata_genes.obs['x'] <= 18000)*(adata_genes.obs['x'] >= 15000)*(adata_genes.obs['y'] <= 14000)*(adata_genes.obs['y'] >= 9000)
ind_tumor2_sub = (adata_genes.obs['x'] <= 18000)*(adata_genes.obs['x'] >= 15000)*(adata_genes.obs['y'] <= 12000)*(adata_genes.obs['y'] >= 10000)

In [ ]:
sc.pl.spatial(adata_raw[ind_tumor2, :], color='EP300--transcript', layer='counts', vmax=1.2, img_key='hires', size=1.5, alpha_img=1.0)

In [ ]:
plt.rcParams['axes.facecolor'] = 'black'
sc.pl.spatial(adata_raw[ind_tumor2_sub, :], color='EP300--transcript', layer='counts', vmax=1.2, img_key='hires', size=1.5, alpha_img=1.0)

In [ ]:
adata_raw.obs['celltype'] = adata_genes.obs['celltype'].copy()
adata_raw.uns['celltype_colors'] = adata_genes.uns['celltype_colors'].copy()

In [ ]:
plt.rcParams['axes.facecolor'] = 'white'
sc.set_figure_params(figsize=(8, 6))
sc.pl.violin(adata_raw, keys='EP300--transcript', groupby='celltype', rotation=90, layer='counts', use_raw=False)

In [ ]:
sc.pl.spatial(adata_raw, color='NOTCH1--transcript', layer='counts', vmax=0.7, img_key='hires', size=1.2, alpha_img=1.0)

In [ ]:
sc.pl.spatial(adata_raw[ind_tumor2, :], color='NOTCH1--transcript', layer='counts', vmax=0.7, img_key='hires', size=1.2, alpha_img=1.0)

In [ ]:
plt.rcParams['axes.facecolor'] = 'white'
sc.set_figure_params(figsize=(8, 6))
sc.pl.violin(adata_raw, keys='NOTCH1--transcript', groupby='celltype', rotation=90, layer='counts', use_raw=False)

### Hallmark gene sets

In [ ]:
hallmark = dc.op.hallmark(organism="human")
hallmark

### Pathway analysis for ROIs

In [ ]:
def hallmark_analysis(adata_genes_sub):
    dc.mt.ulm(data=adata_genes_sub, net=hallmark, layer='counts')
    hallmark_score = dc.pp.get_obsm(adata=adata_genes_sub, key="score_ulm")
    sc.pp.scale(hallmark_score)
    
    hallmark_df = dc.tl.rankby_group(adata=hallmark_score, groupby="celltype", reference="rest", method="t-test_overestim_var")
    hallmark_df = hallmark_df[(hallmark_df["padj"] < 0.05) * (hallmark_df["name"] != 'SPERMATOGENESIS') * (hallmark_df["name"] != 'UV_RESPONSE_UP') * (hallmark_df["name"] != 'UV_RESPONSE_DN')]
    
    n_markers = 3
    hallmark_source_markers = (
        hallmark_df.groupby("group")
        .head(n_markers)
        .drop_duplicates("name")
        .groupby("group")["name"]
        .apply(lambda x: list(x))
        .to_dict()
    )

    return hallmark_score, hallmark_source_markers

In [ ]:
adata_genes_sub = adata_genes[ind_border, :].copy()
hallmark_score_border, hallmark_source_markers_border = hallmark_analysis(adata_genes_sub)
sc.set_figure_params(figsize=(20, 6), fontsize=25)
sc.pl.matrixplot(
    adata=hallmark_score_border,
    var_names=hallmark_source_markers_border,
    groupby="celltype",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
)

In [ ]:
adata_genes_sub = adata_genes[ind_tumor1, :].copy()
hallmark_score_tumor1, hallmark_source_markers_tumor1 = hallmark_analysis(adata_genes_sub)
sc.set_figure_params(figsize=(20, 6), fontsize=25)
sc.pl.matrixplot(
    adata=hallmark_score_tumor1,
    var_names=hallmark_source_markers_tumor1,
    groupby="celltype",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
)

In [ ]:
adata_genes_sub = adata_genes[ind_tumor2, :].copy()
hallmark_score_tumor2, hallmark_source_markers_tumor2 = hallmark_analysis(adata_genes_sub)
sc.set_figure_params(figsize=(20, 6), fontsize=25)
sc.pl.matrixplot(
    adata=hallmark_score_tumor2,
    var_names=hallmark_source_markers_tumor2,
    groupby="celltype",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
)

In [ ]:
adata_genes_sub = adata_genes[ind_tumor3, :].copy()
hallmark_score_tumor3, hallmark_source_markers_tumor3 = hallmark_analysis(adata_genes_sub)
sc.set_figure_params(figsize=(20, 6), fontsize=25)
sc.pl.matrixplot(
    adata=hallmark_score_tumor3,
    var_names=hallmark_source_markers_tumor3,
    groupby="celltype",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
)

In [ ]:
adata_genes_sub = adata_genes[ind_tumor4, :].copy()
hallmark_score_tumor4, hallmark_source_markers_tumor4 = hallmark_analysis(adata_genes_sub)
sc.set_figure_params(figsize=(20, 6), fontsize=25)
sc.pl.matrixplot(
    adata=hallmark_score_tumor4,
    var_names=hallmark_source_markers_tumor4,
    groupby="celltype",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
)

In [ ]:
dc.mt.ulm(data=adata_genes, net=hallmark, layer='counts')

In [ ]:
hallmark_score = dc.pp.get_obsm(adata=adata_genes, key="score_ulm")
hallmark_score

In [ ]:
sc.pp.scale(hallmark_score)

In [ ]:
hallmark_df = dc.tl.rankby_group(adata=hallmark_score, groupby="celltype", reference="rest", method="t-test_overestim_var")
hallmark_df = hallmark_df[(hallmark_df["padj"] < 0.05) * (hallmark_df["name"] != 'SPERMATOGENESIS') * (hallmark_df["name"] != 'UV_RESPONSE_UP') * (hallmark_df["name"] != 'UV_RESPONSE_DN')]
hallmark_df

In [ ]:
n_markers = 3
hallmark_source_markers = (
    hallmark_df.groupby("group")
    .head(n_markers)
    .drop_duplicates("name")
    .groupby("group")["name"]
    .apply(lambda x: list(x))
    .to_dict()
)
hallmark_source_markers

In [ ]:
sc.set_figure_params(figsize=(20, 6), fontsize=25)
sc.pl.matrixplot(
    adata=hallmark_score,
    var_names=hallmark_source_markers,
    groupby="celltype",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
    swap_axes=True,
)